# Analisis dan Prediksi Tingkat Kemiskinan Provinsi
Berdasarkan ketersediaan sarana kesehatan desa/kelurahan (BPS, 38 provinsi).

Cara pakai di Colab: upload 2 file Excel ke panel Files, lalu jalankan cell dari atas ke bawah.

**Catatan:** Prediksi ini indikatif saja, bukan dasar anggaran. MAE Random Forest sekitar 3,87 poin.

## Langkah 2: Load, cleaning, dan gabung data (Bab 3 laporan)
Baca sebagai teks, ambil 39 baris pertama, buang baris Indonesia (total nasional), pilih September 2025 dan 4 sarana, samakan nama kolom, ubah strip jadi 0, ubah teks jadi angka, gabung inner join one_to_one. Hasil yang benar: (38, 8), 0 kosong, total RS 2637, Puskesmas 10280, Pustu 24383, Apotek 17636.

In [ ]:
import pandas as pd
import numpy as np

miskin = pd.read_excel("Jumlah_dan_Persentase_Penduduk_Miskin_Menurut_Provinsi__2025.xlsx", dtype=str).iloc[:39]
faskes = pd.read_excel("Jumlah_Desa_Kelurahan_Yang_Memiliki_Sarana_Kesehatan_Menurut_Provinsi__2024.xlsx", dtype=str).iloc[:39]
miskin = miskin[miskin["Provinsi"] != "Indonesia"]
faskes = faskes[faskes["Provinsi"] != "Indonesia"]
miskin = miskin.rename(columns={
    "Provinsi": "provinsi",
    "Garis Kemiskinan - September (Rp)": "garis_kemiskinan",
    "Jumlah Penduduk Miskin - September (ribu) (Ribu)": "jumlah_miskin_ribu",
    "Persentase Penduduk Miskin - September": "persen_miskin",
})[["provinsi", "garis_kemiskinan", "jumlah_miskin_ribu", "persen_miskin"]]
prefix = "Desa/Kelurahan Yang Memiliki Sarana Kesehatan - "
faskes = faskes.rename(columns={
    "Provinsi": "provinsi",
    prefix + "Rumah Sakit": "desa_rs",
    prefix + "Puskesmas": "desa_puskesmas",
    prefix + "Puskesmas Pembantu": "desa_pustu",
    prefix + "Apotek": "desa_apotek",
})[["provinsi", "desa_rs", "desa_puskesmas", "desa_pustu", "desa_apotek"]]
miskin["provinsi"] = miskin["provinsi"].str.strip()
faskes["provinsi"] = faskes["provinsi"].str.strip()
cols_faskes = ["desa_rs", "desa_puskesmas", "desa_pustu", "desa_apotek"]
faskes[cols_faskes] = faskes[cols_faskes].replace({"–": 0, "-": 0})
faskes[cols_faskes] = faskes[cols_faskes].apply(pd.to_numeric, errors="raise").astype(int)
for c in ["garis_kemiskinan", "jumlah_miskin_ribu", "persen_miskin"]:
    miskin[c] = pd.to_numeric(miskin[c], errors="raise")
df = miskin.merge(faskes, on="provinsi", how="inner", validate="one_to_one")
print(df.shape)
print("Nilai kosong:", int(df.isna().sum().sum()))
print(df.head())
print("Total RS:", df["desa_rs"].sum(), "| Puskesmas:", df["desa_puskesmas"].sum(), "| Pustu:", df["desa_pustu"].sum(), "| Apotek:", df["desa_apotek"].sum())


## Langkah 3: EDA - statistik, korelasi, dan 5 grafik (Bab 4 laporan)
Pearson = hubungan garis lurus, Spearman = hubungan peringkat (lebih tahan outlier). Korelasi sarana dengan kemiskinan lemah dan negatif (-0,15 sampai -0,25), bukan bukti sebab-akibat.

In [ ]:
print(df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]])
q1 = df["persen_miskin"].quantile(0.25)
q3 = df["persen_miskin"].quantile(0.75)
iqr = q3 - q1
print("Q1:", q1, "| Q3:", q3, "| Batas atas:", q3 + 1.5*iqr)
print(df[df["persen_miskin"] > q3 + 1.5*iqr][["provinsi", "persen_miskin"]])
for v in ["desa_rs", "desa_puskesmas", "desa_pustu", "desa_apotek", "garis_kemiskinan", "jumlah_miskin_ribu"]:
    print(v, "| Pearson:", round(df[v].corr(df["persen_miskin"], method="pearson"), 4), "| Spearman:", round(df[v].corr(df["persen_miskin"], method="spearman"), 4))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
top10 = df.nlargest(10, "persen_miskin").sort_values("persen_miskin")
plt.figure(figsize=(8, 5)); plt.barh(top10["provinsi"], top10["persen_miskin"]); plt.title("Top 10 Provinsi dengan Kemiskinan Tertinggi"); plt.xlabel("Persen miskin (%)"); plt.show()
plt.figure(figsize=(8, 5)); plt.hist(df["persen_miskin"], bins=10); plt.axvline(df["persen_miskin"].median(), linestyle="--", label="Median 9,345%"); plt.axvline(df["persen_miskin"].mean(), linestyle=":", label="Mean 10,27%"); plt.title("Distribusi Persen Miskin"); plt.legend(); plt.show()
plt.figure(figsize=(8, 3)); plt.boxplot(df["persen_miskin"], vert=False); plt.title("Boxplot Persen Miskin"); plt.show()
plt.figure(figsize=(8, 5)); plt.scatter(df["desa_puskesmas"], df["persen_miskin"]); plt.title("Puskesmas vs Persen Miskin"); plt.xlabel("Desa ber-Puskesmas"); plt.ylabel("Persen miskin (%)"); plt.show()
plt.figure(figsize=(8, 6)); sns.heatmap(df[["persen_miskin", "desa_rs", "desa_puskesmas", "desa_pustu", "desa_apotek", "garis_kemiskinan"]].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1); plt.title("Heatmap Korelasi"); plt.show()


## Langkah 4: Pemodelan LOOCV - Ridge + Random Forest (Bab 5-6 laporan)
LOOCV: tahan 1 provinsi jadi ujian, latih 37 sisanya, ulangi 38 kali. Fitur hanya 4 sarana, garis kemiskinan dan jumlah miskin TIDAK dipakai di model (leakage). Harapan: Ridge MAE 4,82 / RMSE 6,26 / R2 -0,03; Random Forest MAE sekitar 3,87 / RMSE 5,04 / R2 0,34; Baseline MAE 4,79 / RMSE 6,35 / R2 -0,05.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
feats = ["desa_rs", "desa_puskesmas", "desa_pustu", "desa_apotek"]
X = df[feats].values; y = df["persen_miskin"].values; loo = LeaveOneOut()
def ukur(nama, y_asli, y_pred):
    print(f"{nama} | MAE: {mean_absolute_error(y_asli, y_pred):.4f} | RMSE: {np.sqrt(mean_squared_error(y_asli, y_pred)):.4f} | R2: {r2_score(y_asli, y_pred):.4f}")
ridge = make_pipeline(StandardScaler(), Ridge(alpha=10)); pred_ridge = cross_val_predict(ridge, X, y, cv=loo); ukur("Ridge", y, pred_ridge)
rf = RandomForestRegressor(n_estimators=200, max_depth=3, min_samples_leaf=3, random_state=42); pred_rf = cross_val_predict(rf, X, y, cv=loo); ukur("Random Forest", y, pred_rf)
pred_base = np.array([y[latih].mean() for latih, _ in loo.split(X)]); ukur("Baseline rata-rata", y, pred_base)
err = pd.DataFrame({"provinsi": df["provinsi"].values, "aktual": y, "prediksi": pred_rf}); err["galat"] = (err["aktual"] - err["prediksi"]).abs()
print(err.sort_values("galat", ascending=False).head(8).round(2).to_string(index=False))
rf_full = RandomForestRegressor(n_estimators=200, max_depth=3, min_samples_leaf=3, random_state=42).fit(X, y)
for f, imp in sorted(zip(feats, rf_full.feature_importances_), key=lambda t: -t[1]):
    print(f, ":", round(float(imp), 4))
df["prediksi_rf"] = pred_rf; df["prediksi_ridge"] = pred_ridge


## Langkah 5: Prediksi interaktif (aplikasi di Colab)
Geser slider, prediksi langsung berubah. Model final dilatih di semua 38 provinsi (bukan LOOCV lagi). Contoh: RS 119, Puskesmas 490, Pustu 995, Apotek 1248 -> Random Forest 9,12%, Ridge 9,61%. Papua Tengah (14, 98, 148, 45) aslinya 29,45% tapi model sekitar 17% - contoh keterbatasan.

In [ ]:
from ipywidgets import interact, IntSlider
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
ridge_final = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(X, y)
def prediksi(desa_rs, desa_puskesmas, desa_pustu, desa_apotek):
    data_baru = [[desa_rs, desa_puskesmas, desa_pustu, desa_apotek]]
    print(f"Input: RS={desa_rs}, Puskesmas={desa_puskesmas}, Pustu={desa_pustu}, Apotek={desa_apotek}")
    print(f"Prediksi Random Forest (model terbaik): {rf_full.predict(data_baru)[0]:.2f}%")
    print(f"Prediksi Ridge (pembanding): {ridge_final.predict(data_baru)[0]:.2f}%")
    print("Catatan: ini prediksi indikatif saja. MAE model sekitar 3,87 poin.")
interact(prediksi,
         desa_rs=IntSlider(value=42, min=9, max=346, description="Desa RS"),
         desa_puskesmas=IntSlider(value=208, min=52, max=1110, description="Desa Puskesmas"),
         desa_pustu=IntSlider(value=442, min=147, max=2218, description="Desa Pustu"),
         desa_apotek=IntSlider(value=262, min=8, max=2903, description="Desa Apotek"))
